# Khảo sát dữ liệu mua hàng điện tử

Khảo sát cấu trúc, chất lượng và phạm vi dữ liệu phục vụ phân khúc khách hàng bằng RFM.


In [1]:
from pathlib import Path
from collections import Counter
import csv
import json
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

DATA_PATH = ROOT / 'data/raw/kz.csv'
REPORT_DIR = ROOT / 'reports'
REPORT_DIR.mkdir(exist_ok=True)


## 1. Xác định các cột của file

Đọc tiêu đề để xác định tên cột và số trường kỳ vọng cho mỗi bản ghi.


In [2]:
with DATA_PATH.open(encoding='utf-8-sig', newline='') as stream:
    header = next(csv.reader(stream))

expected_fields = len(header)
print('Expected fields per record:', expected_fields)
display(pd.DataFrame({'position': range(1, expected_fields + 1), 'column': header}))


Expected fields per record: 8


,position,column
0,1,event_time
1,2,order_id
2,3,product_id
3,4,category_id
4,5,category_code
5,6,brand
6,7,price
7,8,user_id


## 2. Thống kê số trường trên từng bản ghi

Đếm số giá trị thực tế trên toàn bộ file và so sánh với số cột trong tiêu đề.
Bảng thống kê cho biết nhóm nào đủ trường, thiếu trường hoặc thừa trường trước khi chọn mẫu để kiểm tra.


In [3]:
field_counts = Counter()
incomplete_rows = []

with DATA_PATH.open(encoding='utf-8-sig', newline='') as stream:
    reader = csv.reader(stream)
    next(reader)
    for index, row in enumerate(reader):
        field_counts[len(row)] += 1
        if len(row) != expected_fields:
            incomplete_rows.append(index)

total_rows = sum(field_counts.values())
structure = pd.DataFrame([
    {
        'field_count': size,
        'rows': count,
        'pct': round(count / total_rows * 100, 2),
        'status': 'expected' if size == expected_fields else 'too_few' if size < expected_fields else 'too_many',
    }
    for size, count in sorted(field_counts.items())
])
display(structure)


,field_count,rows,pct,status
0,6,431954,16.4,too_few
1,8,2201567,83.6,expected


### 2.1. Quan sát nhóm thiếu trường

Thống kê phía trên xác định nhóm 6 trường cần kiểm tra. Trích ví dụ từ nhóm này và đối chiếu với
mẫu đủ trường để xem vị trí giá trị. `value_1` đến `value_6` là vị trí trong dòng gốc;
`csv_record` là thứ tự bản ghi sau tiêu đề. Chọn ví dụ có và thiếu mã ở vị trí cuối.


In [4]:
short_records = []
examples_with_customer = []
examples_without_customer = []

with DATA_PATH.open(encoding='utf-8-sig', newline='') as stream:
    reader = csv.reader(stream)
    next(reader)
    for index, row in enumerate(reader):
        if len(row) != 6:
            continue
        short_records.append((index, row[3], row[4], row[5]))
        example = {'csv_record': index + 1}
        example.update({f'value_{position}': value or '(empty)'
                        for position, value in enumerate(row, 1)})
        if row[5].strip() and len(examples_with_customer) < 2:
            examples_with_customer.append(example)
        elif not row[5].strip() and len(examples_without_customer) < 1:
            examples_without_customer.append(example)

examples = pd.DataFrame(examples_with_customer + examples_without_customer)
raw = pd.read_csv(DATA_PATH, dtype='string', keep_default_na=False)
if list(raw.columns) != header or len(raw) != total_rows:
    raise ValueError('The loaded CSV does not match the structure scan.')

print('Sample records with all expected fields:')
display(raw.drop(index=incomplete_rows).head(2))
print('Sample records from the six-field group:')
display(examples)


Sample records with all expected fields:


,event_time,order_id,product_id,category_id,category_code,brand,price,user_id
0,2020-04-24 11:50:39 UTC,2294359932054536986,1515966223509089906,2268105426648170900,electronics.tablet,samsung,162.01,1515915625441993984
1,2020-04-24 11:50:39 UTC,2294359932054536986,1515966223509089906,2268105426648170900,electronics.tablet,samsung,162.01,1515915625441993984


Sample records from the six-field group:


,csv_record,value_1,value_2,value_3,value_4,value_5,value_6
0,1813,2020-05-02 11:12:20 UTC,2300138854784434309,1515966223509353669,(empty),0.02,1515915625453675584
1,3634,2020-05-06 07:37:55 UTC,2302930038149874117,1515966223509128764,(empty),38.17,1515915625455624044
2,28819,1970-01-01 00:33:40 UTC,2340102743319904671,1515966223523303300,(empty),13.87,(empty)


### 2.2. Kiểm tra mẫu và xử lý có điều kiện

Kiểm tra toàn bộ dòng 6 trường theo ba điều kiện: vị trí 4 trống, vị trí 5 có dạng số thập phân,
vị trí 6 là dãy chữ số hoặc trống.

Với dòng khớp mẫu, giữ ba giá trị đầu, chuyển hai giá trị cuối về `price` và `user_id`,
để các cột danh mục và thương hiệu trống. Đánh dấu dòng đã sửa và hiển thị mẫu trước–sau.
Dòng không khớp được tách riêng. Quy tắc dựa trên cấu trúc quan sát, các giá trị vẫn cần được kiểm tra ở bước 4.


In [5]:
short = pd.DataFrame(short_records, columns=['row_index', 'fourth_value', 'price', 'user_id'])
for column in ['fourth_value', 'price', 'user_id']:
    short[column] = short[column].str.strip()

blank_fourth = short['fourth_value'].eq('')
price_like = short['price'].str.fullmatch(r'-?\d+(\.\d+)?', na=False)
customer_like = short['user_id'].str.fullmatch(r'\d*', na=False)
matches_pattern = blank_fourth & price_like & customer_like

pattern_counts = {
    'six_field_rows': len(short),
    'blank_fourth_value': int(blank_fourth.sum()),
    'numeric_fifth_value': int(price_like.sum()),
    'numeric_or_empty_sixth_value': int(customer_like.sum()),
    'matching_all_conditions': int(matches_pattern.sum()),
    'not_matching': int((~matches_pattern).sum()),
}
display(pd.Series(pattern_counts, name='rows').to_frame())

repair = short.loc[matches_pattern].set_index('row_index')
row_status = pd.Series('full_width', index=raw.index)
row_status.loc[incomplete_rows] = 'unmatched_structure'
row_status.loc[repair.index] = 'repaired_6_fields'

audit = raw.loc[row_status.ne('unmatched_structure')].copy()
audit.loc[repair.index, ['category_id', 'category_code', 'brand']] = ''
audit.loc[repair.index, 'price'] = repair['price'].astype('string')
audit.loc[repair.index, 'user_id'] = repair['user_id'].astype('string')
audit = audit.apply(lambda column: column.str.strip().replace('', pd.NA))

example_indices = examples['csv_record'].sub(1).tolist()
print('Before: columns assigned by the default reader')
display(raw.loc[example_indices].fillna('(empty)').replace('', '(empty)'))
print('After: columns assigned using the observed pattern')
display(audit.loc[example_indices].fillna('(empty)'))
display(row_status.value_counts().to_frame('rows'))


,rows
six_field_rows,431954
blank_fourth_value,431954
numeric_fifth_value,431954
numeric_or_empty_sixth_value,431954
matching_all_conditions,431954
not_matching,0


Before: columns assigned by the default reader


,event_time,order_id,product_id,category_id,category_code,brand,price,user_id
1812,2020-05-02 11:12:20 UTC,2300138854784434309,1515966223509353669,(empty),0.02,1515915625453675584,(empty),(empty)
3633,2020-05-06 07:37:55 UTC,2302930038149874117,1515966223509128764,(empty),38.17,1515915625455624044,(empty),(empty)
28818,1970-01-01 00:33:40 UTC,2340102743319904671,1515966223523303300,(empty),13.87,(empty),(empty),(empty)


After: columns assigned using the observed pattern


,event_time,order_id,product_id,category_id,category_code,brand,price,user_id
1812,2020-05-02 11:12:20 UTC,2300138854784434309,1515966223509353669,(empty),(empty),(empty),0.02,1515915625453675584
3633,2020-05-06 07:37:55 UTC,2302930038149874117,1515966223509128764,(empty),(empty),(empty),38.17,1515915625455624044
28818,1970-01-01 00:33:40 UTC,2340102743319904671,1515966223523303300,(empty),(empty),(empty),13.87,(empty)


,rows
full_width,2201567
repaired_6_fields,431954


## 3. Thống kê dữ liệu thiếu và bản ghi trùng

Thống kê số ô trống, tỷ lệ thiếu và số giá trị phân biệt trên dữ liệu sau xử lý cấu trúc.
Đếm bản ghi giống nhau sau khi bỏ khoảng trắng đầu/cuối; chưa loại vì có thể là nhiều đơn vị cùng sản phẩm được mua.


In [6]:
quality = pd.DataFrame({
    'missing_count': audit.isna().sum(),
    'missing_pct': (audit.isna().mean() * 100).round(2),
    'distinct_count': audit.nunique(),
})
quality.index.name = 'column'
duplicate_rows = int(audit.duplicated().sum())

display(quality)
print('Repeated rows after trimming:', duplicate_rows)


,missing_count,missing_pct,distinct_count
column,,,
event_time,0,0.00,1316174
order_id,0,0.00,1435266
product_id,0,0.00,25113
category_id,431954,16.40,927
category_code,1044156,39.65,123
brand,544624,20.68,908
price,0,0.00,2896
user_id,2030733,77.11,234745


Repeated rows after trimming: 675


## 4. Kiểm tra tính hợp lệ

Kiểm tra ngày thuộc năm 2020, giá dương, định dạng mã khách và mã đơn; phát hiện đơn gắn với nhiều khách.
Các lỗi được đếm riêng, một bản ghi có thể thuộc nhiều nhóm lỗi.


In [7]:
dates = pd.to_datetime(audit['event_time'],
                       format='%Y-%m-%d %H:%M:%S UTC', errors='coerce', utc=True)
prices = pd.to_numeric(audit['price'], errors='coerce')

valid_date = dates.notna() & dates.dt.year.eq(2020)
valid_price = prices.notna() & prices.gt(0)
valid_customer = audit['user_id'].str.fullmatch(r'\d+', na=False)
valid_order = audit['order_id'].str.fullmatch(r'\d+', na=False)

pairs = audit.loc[valid_customer & valid_order, ['order_id', 'user_id']]
customers_per_order = pairs.groupby('order_id')['user_id'].nunique()
conflicting_orders = customers_per_order[customers_per_order.gt(1)].index
order_conflict = audit['order_id'].isin(conflicting_orders)

checks = {
    'invalid_or_missing_date': int(dates.isna().sum()),
    'date_outside_2020': int((dates.notna() & ~valid_date).sum()),
    'invalid_or_missing_price': int(prices.isna().sum()),
    'nonpositive_price': int(prices.le(0).sum()),
    'invalid_or_missing_customer_id': int((~valid_customer).sum()),
    'invalid_or_missing_order_id': int((~valid_order).sum()),
    'orders_with_multiple_customers': int(len(conflicting_orders)),
}
display(pd.Series(checks, name='count').to_frame())
display(dates.dt.year.value_counts(dropna=False).sort_index().to_frame('rows'))


,count
invalid_or_missing_date,0
date_outside_2020,19631
invalid_or_missing_price,0
nonpositive_price,3842
invalid_or_missing_customer_id,2030733
invalid_or_missing_order_id,0
orders_with_multiple_customers,0


,rows
event_time,
1970,19631
2020,2613890


## 5. Xác định phạm vi phân tích

Chọn bản ghi đáp ứng các kiểm tra; thống kê số khách, số đơn, tỷ lệ mua lại và thời gian quan sát.
Đếm riêng các dòng đủ cột ban đầu và các dòng dùng thêm sau sửa để thể hiện ảnh hưởng của quy tắc.


In [8]:
eligible = valid_date & valid_price & valid_customer & valid_order & ~order_conflict
selected = audit.loc[eligible]
order_counts = selected.groupby('user_id')['order_id'].nunique()
repeat_customers = int(order_counts.ge(2).sum())

summary = {
    'raw_rows': len(raw),
    'incomplete_rows': len(incomplete_rows),
    'full_width_rows': int(row_status.eq('full_width').sum()),
    'repaired_rows': len(repair),
    'unmatched_structure_rows': int(row_status.eq('unmatched_structure').sum()),
    'eligible_full_width_rows': int((eligible & row_status.loc[audit.index].eq('full_width')).sum()),
    'eligible_repaired_rows': int((eligible & row_status.loc[audit.index].eq('repaired_6_fields')).sum()),
    'eligible_rows': len(selected),
    'row_coverage_pct': round(len(selected) / len(raw) * 100, 2),
    'customers': len(order_counts),
    'orders': int(selected['order_id'].nunique()),
    'repeat_customers': repeat_customers,
    'repeat_customer_pct': round(repeat_customers / len(order_counts) * 100, 2) if len(order_counts) else None,
    'first_purchase': dates.loc[eligible].min().isoformat() if eligible.any() else None,
    'last_purchase': dates.loc[eligible].max().isoformat() if eligible.any() else None,
    'reference_date': (dates.loc[eligible].max().normalize() + pd.Timedelta(days=1)).isoformat() if eligible.any() else None,
}
display(pd.Series(summary, name='value').to_frame())


,value
raw_rows,2633521
incomplete_rows,431954
full_width_rows,2201567
repaired_rows,431954
unmatched_structure_rows,0
eligible_full_width_rows,562823
eligible_repaired_rows,37961
eligible_rows,600784
row_coverage_pct,22.81
customers,234359


## 6. Xuất báo cáo khảo sát

Lưu thống kê cấu trúc, mẫu trước–sau, chất lượng từng cột và số liệu tổng hợp vào `reports/`.


In [9]:
structure.to_csv(REPORT_DIR / 'row_structure.csv', index=False, encoding='utf-8-sig')
quality.to_csv(REPORT_DIR / 'column_quality.csv', encoding='utf-8-sig')
example_indices = examples['csv_record'].sub(1).tolist()
before = raw.loc[example_indices].copy()
before.insert(0, 'csv_record', examples['csv_record'].tolist())
after = audit.loc[example_indices].copy()
after.insert(0, 'csv_record', examples['csv_record'].tolist())
before.to_csv(REPORT_DIR / 'short_rows_before.csv', index=False, encoding='utf-8-sig')
after.to_csv(REPORT_DIR / 'short_rows_after.csv', index=False, encoding='utf-8-sig')
report = {
    'summary': summary,
    'observed_pattern': pattern_counts,
    'checks_on_full_and_repaired_rows': checks,
    'repeated_rows_after_alignment_and_trimming': duplicate_rows,
    'rows_by_field_count': dict(field_counts),
    'scope': 'Full-width rows plus pattern-matched short rows; duplicate removal is not applied.',
}
(REPORT_DIR / 'data_audit.json').write_text(json.dumps(report, indent=2), encoding='utf-8')
print('Saved reports. The original CSV was not changed.')


Saved reports. The original CSV was not changed.
